# 03 · 策略研究与过拟合

> **学习目标**
> 1. 学会用**参数曲面**判断一个策略是「真优势」还是「运气」
> 2. 掌握**前向滚动验证**，得到接近无偏的样本外业绩
> 3. 亲手复现「样本内赚钱、样本外亏钱」——量化研究里最重要的一课

> **一句话**：这一节的核心不是找最优参数，而是**学会怀疑自己的回测结果**。

---

### 三种典型的假象

| 假象 | 表现 | 成因 |
|---|---|---|
| 数据窥探偏差 | 试了 200 组参数，挑了最好的那组 | 你其实在拟合噪声 |
| 参数尖峰 | 只有 `fast=7, slow=43` 赚钱，旁边全亏 | 参数恰好对齐了某几段行情 |
| 样本外失效 | 样本内 Sharpe 1.5，实盘 -0.3 | 优势根本不存在 |

In [ ]:
import warnings

warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd

from qlearn.backtest import BacktestEngine
from qlearn.data import load_panel, make_synthetic_panel
from qlearn.research import grid_search, split_in_out_sample, walk_forward
from qlearn.strategies import create_strategy
from qlearn.utils import plot_param_surface

pd.set_option('display.width', 190)
print('环境就绪')

In [ ]:
USE_REAL_DATA = True
SYMBOLS = ['600519', '000858', '601318', '600036', '000001']
START, END = '2019-01-01', '2024-12-31'

try:
    panel = load_panel(SYMBOLS, start=START, end=END) if USE_REAL_DATA else None
    if panel is None:
        raise RuntimeError('已关闭真实数据')
    data_source = '真实 A 股'
except Exception as exc:
    print(f'[降级] {type(exc).__name__}: {str(exc)[:100]}')
    panel = make_synthetic_panel(SYMBOLS, start=START, end=END, seed=42)
    data_source = '合成数据（无现实意义）'

engine = BacktestEngine(initial_capital=1_000_000.0)
print(f'数据源: {data_source}｜{panel.n_symbols} 个标的 x {panel.n_dates} 个交易日')

## 1. 参数扫描：看的是曲面，不是最优点

**绝对不要**做这件事：跑 200 组参数，把 Sharpe 最高的那组拿去实盘。
这叫数据窥探偏差——你只是找到了拟合噪声最厉害的那组参数。

**要做**的是：看整张参数曲面长什么样。

- **高原**：一片连续参数都能赚钱 → 存在真实的结构性特征，可以继续研究
- **尖峰**：只有一个格子突出，四周为负 → 过拟合，样本外必然失效
- **悬崖**：相邻格子结果剧烈反转 → 参数选择风险极高，不能实盘

In [ ]:
GRID = {'fast': [5, 10, 15, 20], 'slow': [30, 45, 60, 90, 120]}

table = grid_search('ma_cross', GRID, panel, engine=engine, sort_by='年化Sharpe')

print(f'共评估 {len(table)} 组参数（数据源：{data_source}）')
display_cols = ['fast', 'slow', '累计收益', '最大回撤', '年化Sharpe', 'Calmar比率', '年化双边换手率']
view = table[display_cols].copy()
for col in ['累计收益', '最大回撤', '年化双边换手率']:
    view[col] = view[col].map(lambda v: f'{v:.2%}')
for col in ['年化Sharpe', 'Calmar比率']:
    view[col] = view[col].map(lambda v: f'{v:.3f}')

print('按 Sharpe 排序前 10 组:')
display(view.head(10))

In [ ]:
fig, ax = plot_param_surface(table, 'fast', 'slow', metric='年化Sharpe', figsize=(8, 6))
plt.show()

### 怎么读上面这张图？

1. **最优格子旁边的四个格子**是正值还是负值？
   - 四个都是正 → 高原，参数稳健
   - 三个是负 → 尖峰，过拟合
2. **换手率那一列**是否随参数剧烈变化？
   - 短周期参数换手率会高一个数量级，成本拖累完全不同
3. 如果整张图**几乎全是负数** → 这个策略在这个股票池上没有优势，
   不要试图通过调参救它。

In [ ]:
surface = table.pivot_table(index='fast', columns='slow', values='年化Sharpe')
print('Sharpe 参数矩阵:')
display(surface.round(3))

positive_ratio = (table['年化Sharpe'] > 0).mean()
print()
print(f'正 Sharpe 的参数组合占比: {positive_ratio:.1%}')
print(f'Sharpe 极差: {table["年化Sharpe"].max() - table["年化Sharpe"].min():.3f}')
print()
print('极差越大，说明策略表现对参数越敏感，实盘越危险。')

## 2. 最简单的样本外检验：一刀切

把数据切成「前 60% 训练 / 后 40% 测试」，只在训练集上看结果，
测试集**从头到尾只看一次**。

In [ ]:
split_date = panel.dates[int(len(panel.dates) * 0.6)]
in_sample, out_sample = split_in_out_sample(panel, split_date)
print(f'样本内: {in_sample.dates[0]:%Y-%m-%d} ~ {in_sample.dates[-1]:%Y-%m-%d}（{in_sample.n_dates} 天）')
print(f'样本外: {out_sample.dates[0]:%Y-%m-%d} ~ {out_sample.dates[-1]:%Y-%m-%d}（{out_sample.n_dates} 天）')

# 只在样本内做参数寻优
is_table = grid_search('ma_cross', GRID, in_sample, engine=engine, sort_by='年化Sharpe')
best = is_table.iloc[0]
best_params = {'fast': int(best['fast']), 'slow': int(best['slow'])}
print(f'\n样本内最优参数: {best_params}')

# 把参数原封不动搬到样本外
strategy = create_strategy('ma_cross', **best_params)
res_is = engine.run(strategy.generate_weights(in_sample), in_sample, label='样本内')
res_oos = engine.run(strategy.generate_weights(out_sample), out_sample, label='样本外')

compare = pd.DataFrame(
    {
        '样本内': res_is.metrics(),
        '样本外': res_oos.metrics(),
    }
)
print()
display(compare.loc[['累计收益', '年化收益(CAGR)', '年化Sharpe', '最大回撤', '年化双边换手率']].round(4))

## 3. 前向滚动验证：模拟真实的策略迭代流程

一刀切只有一个样本外区间，运气成分仍然很大。前向滚动验证把它扩展成多折：

```
|---- 训练 1 ----|-- 测试 1 --|
          |---- 训练 2 ----|-- 测试 2 --|
                    |---- 训练 3 ----|-- 测试 3 --|
```

1. 在训练窗口上选参数，把参数**原封不动**应用到紧随其后的测试窗口
2. 所有测试窗口的收益首尾相接，得到一条**近乎无偏**的样本外净值曲线
3. 这才是评价策略的正确口径——样本内指标只能说明优化过程有多过拟合

In [ ]:
wf = walk_forward(
    'ma_cross',
    GRID,
    panel,
    engine=engine,
    train_days=504,   # 约 2 年训练
    test_days=126,    # 约半年测试
    warmup_days=120,  # 测试窗口额外向前取 120 天做指标预热
    sort_by='年化Sharpe',
)
print(repr(wf))

In [ ]:
print('各折明细（IS = 样本内，OOS = 样本外）:')
folds = wf.folds.copy()
for col in ('train_start', 'train_end', 'test_start', 'test_end'):
    folds[col] = folds[col].map(lambda d: f'{d:%Y-%m-%d}')
display(folds)

In [ ]:
gap = wf.overfit_gap()
print('过拟合诊断:')
display(gap)

print()
print('判读标准:')
print('  1. 样本内平均 Sharpe 与样本外 Sharpe 差距很大 -> 严重过拟合')
print('  2. 样本内 Sharpe 本身就很低          -> 策略没有优势，不必谈样本外')
print('  3. 两者接近且都为正                  -> 参数稳健，值得进一步研究')

In [ ]:
equity = wf.oos_equity
curve = equity / equity.iloc[0]
drawdown = curve / curve.cummax() - 1

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(12, 7), sharex=True,
    gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.08},
)
ax1.plot(curve.index, curve, color='#1f4e79', linewidth=1.6,
         label='样本外净值（各折拼接）')
ax1.axhline(1.0, color='black', linewidth=0.8, linestyle=':')
ax1.set_title(f'前向滚动验证的样本外净值｜数据源：{data_source}')
ax1.set_ylabel('净值')
ax1.legend(frameon=False)
ax1.grid(alpha=0.3, linestyle='--')
ax1.spines[['top', 'right']].set_visible(False)

ax2.fill_between(drawdown.index, drawdown.to_numpy(), 0, color='#c0392b', alpha=0.35, linewidth=0)
ax2.plot(drawdown.index, drawdown, color='#c0392b', linewidth=0.9)
ax2.set_ylabel('回撤')
ax2.yaxis.set_major_formatter(lambda v, _: f'{v:.0%}')
ax2.grid(alpha=0.3, linestyle='--')
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

## 4. 对照实验：策略只在「有结构」的市场里才可能赚钱

用合成数据的 `autocorrelation` 旋钮构造三种市场，检验策略是否**如理论预期**地表现：

| 市场 | 自相关 | 预期有效的策略 |
|---|---|---|
| 随机游走 | 0 | 无（扣费后全部亏损） |
| 趋势市 | +0.08 | 动量 / 趋势 |
| 反转市 | −0.08 | 均值回归 |

> 这是一个极其有用的思维习惯：**先问「这个策略赚的是谁的钱」，
> 再用数据验证它是否真的在那个结构下表现更好。**
>
> 为了剥离市场本身的涨跌，下面统一看**相对买入持有的超额收益**。

In [ ]:
CASES = [
    ('随机游走 φ=0', 0.0),
    ('趋势市 φ=+0.08', 0.08),
    ('反转市 φ=-0.08', -0.08),
]
STRATEGIES = [
    ('ma_cross', {'fast': 5, 'slow': 20}),
    ('momentum', {'lookback': 40, 'top_n': 2, 'rebalance_days': 10}),
    ('mean_reversion', {'window': 20, 'entry_z': -1.5}),
]

rows = []
for regime, phi in CASES:
    synth = make_synthetic_panel(SYMBOLS, start=START, end=END, seed=11, autocorrelation=phi)
    bench = engine.run(
        create_strategy('buy_and_hold').generate_weights(synth), synth, label='bench'
    )
    bench_return = bench.metrics()['累计收益']

    for name, params in STRATEGIES:
        strat = create_strategy(name, **params)
        res = engine.run(strat.generate_weights(synth), synth, label=strat.label)
        m = res.metrics()
        rows.append(
            {
                '市场': regime,
                '策略': name,
                '累计收益': m['累计收益'],
                '买入持有': bench_return,
                '超额收益': m['累计收益'] - bench_return,
                '年化Sharpe': m['年化Sharpe'],
                '最大回撤': m['最大回撤'],
            }
        )

result_table = pd.DataFrame(rows)
for col in ('累计收益', '买入持有', '超额收益', '最大回撤'):
    result_table[col] = result_table[col].map(lambda v: f'{v:+.2%}')
result_table['年化Sharpe'] = result_table['年化Sharpe'].map(lambda v: f'{v:+.2f}')

print('合成数据对照实验（超额收益 = 策略 − 买入持有）:')
result_table

### 从这张表该看出什么

1. **随机游走那一行**：所有策略的超额收益都不该显著为正。
   如果有策略在纯随机数据上赚很多钱，**先怀疑代码而不是庆祝**
   ——这是检验回测引擎有没有造假（前视偏差、成本漏算）的最佳基准
2. **趋势市**：动量与双均线应当改善，均值回归应当变差
3. **反转市**：均值回归应当改善

如果结果与理论预期方向相反，说明要么引擎有问题，要么你对策略的理解有问题。

## 5. 小结与练习

**记住五句话**

1. 参数扫描的产出是**曲面**，不是最优点
2. 只有样本外（尤其是前向滚动）的业绩才值得相信
3. 样本内与样本外的巨大落差，就是过拟合的量化度量
4. 在随机游走数据上赚钱 = 代码有 bug
5. 换手率必须与收益一起看，高换手的高收益在扣费后经常不复存在

**动手练习**

1. 把 `train_days` / `test_days` 改成 252/63（更短的窗口，更频繁地更新参数），
   观察样本外结果变好还是变坏。为什么？
2. 把 `sort_by` 从 `年化Sharpe` 换成 `累计收益`，看选出的参数是否更不稳定
3. 把 `GRID` 扩大一倍（比如 `fast` 取 5,7,9,...,25），观察最优参数是否会漂移
   ——参数网格越细，越容易找到「尖峰」，这就是数据窥探偏差的放大过程
4. 把 `walk_forward` 的 `warmup_days` 改成 0，观察样本外结果的变化，
   理解「预热期不裁剪会重复计入收益」这件事
5. **终极练习**：把 `SORT_BY` 换成某个你真正想优化的指标，
   然后问自己：这个指标的提升，是来自真实的优势，还是来自对历史的记忆？